First all the plotting is commented out to make sure this doesn't affect the calculation speeds.

We then start with using the cProfiler to find out about which part of the programm takes the most time.

In [2]:
!python -m cProfile -s cumulative volumerender.py

Rendering Scene 1 of 10.

Rendering Scene 2 of 10.

Rendering Scene 3 of 10.

Rendering Scene 4 of 10.

Rendering Scene 5 of 10.

Rendering Scene 6 of 10.

Rendering Scene 7 of 10.

Rendering Scene 8 of 10.

Rendering Scene 9 of 10.

Rendering Scene 10 of 10.

         2072176 function calls (2032106 primitive calls) in 12.063 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    745/1    0.008    0.000   12.064   12.064 {built-in method builtins.exec}
        1    0.001    0.001   12.064   12.064 volumerender.py:1(<module>)
        1    0.618    0.618   10.217   10.217 volumerender.py:40(main)
       10    0.135    0.013    6.830    0.683 _rgi.py:645(interpn)
       10    0.000    0.000    6.685    0.668 _rgi.py:375(__call__)
       10    4.370    0.437    4.371    0.437 _rgi.py:520(_evaluate_linear)
       95    0.003    0.000    3.641    0.038 __init__.py:1(<module>)
     1800    2.634    0.001    2.634    0.001 volumere

It seems like especially the function "interpn" is a bottleneck for the performance, since more than half of the time is spent inside this function.

But also the transfer-function itself uses quite some time (2.6s). In order to get deeper insisghts into what is happening we also use the lineprofiler on the main function:

In [8]:
!python -m kernprof -l volumerender.py

Rendering Scene 1 of 10.

Rendering Scene 2 of 10.

Rendering Scene 3 of 10.

Rendering Scene 4 of 10.

Rendering Scene 5 of 10.

Rendering Scene 6 of 10.

Rendering Scene 7 of 10.

Rendering Scene 8 of 10.

Rendering Scene 9 of 10.

Rendering Scene 10 of 10.

Wrote profile results to 'volumerender.py.lprof'
Inspect results with:
python -m line_profiler -rmt volumerender.py.lprof


In [9]:
!python -m line_profiler -rmt volumerender.py.lprof

Timer unit: 1e-06 s

Total time: 11.3702 s
File: volumerender.py
Function: main at line 38

Line #      Hits         Time  Per Hit   % Time  Line Contents
    38                                           @profile
    39                                           def main():
    40                                               """Volume Rendering"""
    41                                           
    42                                               # Load Datacube
    43         1       2273.0   2273.0      0.0      f = h5.File("datacube.hdf5", "r")
    44         1      14591.0  14591.0      0.1      datacube = np.array(f["density"])
    45                                           
    46                                               # Datacube Grid
    47         1          3.0      3.0      0.0      Nx, Ny, Nz = datacube.shape
    48         1        106.0    106.0      0.0      x = np.linspace(-Nx / 2, Nx / 2, Nx)
    49         1         16.0     16.0      0.0      y = np.linspac

Here we can also see that the interpolation is the bottle neck and we can also see that the transfer-function takes 26% of the time.
Now let's move the profile decorator to the transferfunction to get more insights on what is happening there as well:

In [6]:
!python -m kernprof -l volumerender.py

Rendering Scene 1 of 10.

Rendering Scene 2 of 10.

Rendering Scene 3 of 10.

Rendering Scene 4 of 10.

Rendering Scene 5 of 10.

Rendering Scene 6 of 10.

Rendering Scene 7 of 10.

Rendering Scene 8 of 10.

Rendering Scene 9 of 10.

Rendering Scene 10 of 10.

Wrote profile results to 'volumerender.py.lprof'
Inspect results with:
python -m line_profiler -rmt volumerender.py.lprof


In [7]:
!python -m line_profiler -rmt volumerender.py.lprof

Timer unit: 1e-06 s

Total time: 3.07087 s
File: volumerender.py
Function: transferFunction at line 14

Line #      Hits         Time  Per Hit   % Time  Line Contents
    14                                           @profile
    15                                           def transferFunction(x):
    16      1800        375.0      0.2      0.0      r = (
    17      5400     267888.0     49.6      8.7          1.0 * np.exp(-((x - 9.0) ** 2) / 1.0)
    18      1800     226855.0    126.0      7.4          + 0.1 * np.exp(-((x - 3.0) ** 2) / 0.1)
    19      1800     233788.0    129.9      7.6          + 0.1 * np.exp(-((x - -3.0) ** 2) / 0.5)
    20                                               )
    21      1800        360.0      0.2      0.0      g = (
    22      5400     273602.0     50.7      8.9          1.0 * np.exp(-((x - 9.0) ** 2) / 1.0)
    23      1800     235744.0    131.0      7.7          + 1.0 * np.exp(-((x - 3.0) ** 2) / 0.1)
    24      1800     229532.0    127.5      7.

Now we know what causes the most delay and is the performance bottleneck in the code. Before working on an optimization we also want to know how well the code performs in terms of memory. For this we use the memory profiler (with decorator around main function).

In [10]:
!python -m memory_profiler volumerender.py

/Users/timeberle/GitHub/volumerender-python/.venv/lib/python3.14/site-packages/memory_profiler.py:752: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  if iscoroutinefunction(func):
Rendering Scene 1 of 10.

Rendering Scene 2 of 10.

Rendering Scene 3 of 10.

Rendering Scene 4 of 10.

Rendering Scene 5 of 10.

Rendering Scene 6 of 10.

Rendering Scene 7 of 10.

Rendering Scene 8 of 10.

Rendering Scene 9 of 10.

Rendering Scene 10 of 10.

Filename: volumerender.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    38  137.953 MiB  137.953 MiB           1   @profile
    39                                         def main():
    40                                             """Volume Rendering"""
    41                                         
    42                                             # Load Datacube
    43  138.984 MiB    1.031 MiB           1       f = h5.File("da

As we can see, we are copying the data twice (first: meshgrid, then: reshaping into qi), which takes some space and might not be necessary.
We also can see that the interpolation line uses by far the most space with 734 MB. So also the memory profiler hints that we should try to optimize the interpolation function.